In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import os 
import nltk

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
!pip install -q pyicu
!pip install -q pycld2
!pip install -q polyglot
!pip install -q textstat
!pip install -q googletrans

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [66 lines of output]
      (running 'icu-config --version')
      (running 'pkg-config --modversion icu-i18n')
      Traceback (most recent call last):
        File "<string>", line 89, in <module>
        File "<frozen os>", line 714, in __getitem__
      KeyError: 'ICU_VERSION'
      
      During handling of the above exception, another exception occurred:
      
      Traceback (most recent call last):
        File "<string>", line 92, in <module>
        File "<string>", line 19, in check_output
        File "C:\Users\CANDRAGNAWN\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 466, in check_output
          return run(*popenargs, stdout=PIPE, timeout=timeout, check=True,
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\CANDRAGNAWN\AppData\Local\Programs\Python\Python312\Lib\subprocess.py"

In [4]:
DATA_PATH = "dataset/"
os.listdir(DATA_PATH)

['sample_submission.csv', 'test.csv', 'test_labels.csv', 'train.csv']

In [15]:
TEST_PATH = os.path.join(DATA_PATH, "test.csv")
TEST_LABELS_PATH = os.path.join(DATA_PATH, "test.labels.csv")
SAMPLE_PATH = os.path.join(DATA_PATH, "sample_submission.csv")
TRAIN_PATH = os.path.join(DATA_PATH, "train.csv")
test_data = pd.read_csv(TEST_PATH)
train_data = pd.read_csv(TRAIN_PATH)
test_labels = pd.read_csv(TEST_LABELS_PATH) if os.path.exists(TEST_LABELS_PATH) else None

In [14]:
df = pd.read_csv('dataset/train.csv')
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


EXPLORATORY DATA ANALYSIS

In [25]:
print(df.columns)

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate', 'label'],
      dtype='str')


In [26]:
from EDA.exploratory_data import exploratory_data_analysis
df['label'] = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].max(axis=1)
exploratory_data_analysis(df)

Data Shape: (159571, 9)

Data Types:
 id                 str
comment_text       str
toxic            int64
severe_toxic     int64
obscene          int64
threat           int64
insult           int64
identity_hate    int64
label            int64
dtype: object

Missing Values:
 id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
label            0
dtype: int64

Statistical Summary:
                toxic   severe_toxic        obscene         threat  \
count  159571.000000  159571.000000  159571.000000  159571.000000   
mean        0.095844       0.009996       0.052948       0.002996   
std         0.294379       0.099477       0.223931       0.054650   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.000000       0.000000       0.000000       0.000000   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.000000       0.0000

,toxic,severe_toxic,obscene,threat,insult,identity_hate,label
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805,0.101679
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420,0.302226
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [7]:
from prepocessing.text_preprocessing import TextPreprocessor
from prepocessing.text_preprocessing import TextPreprocessorNonStopword
preprocessor = TextPreprocessor()
preprocessor_nonstopword = TextPreprocessorNonStopword()
df['clean_comment_text'] = df['comment_text'].apply(
    preprocessor.preprocess_text
)

df['nonstopword_comment_text'] = df['comment_text'].apply(
    preprocessor_nonstopword.preprocess_text_non_stopword
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\CANDRAGNAWN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading tokenizers/punkt/english.pickle: Package
[nltk_data]     'tokenizers/punkt/english.pickle' not found in index


Labelin Data

In [8]:
#definisi kolom di dataset
label_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# jika ada di dalam kolom itu nilai nya 1 pasti toxic
df['is_toxic'] = df[label_columns].max(axis=1)

X1 = df['clean_comment_text']
X2 = df['nonstopword_comment_text']
Y = df['is_toxic']

print("Distribusi Target Y (0 = Non-Toxic, 1 = Toxic):")
print(Y.value_counts())

Distribusi Target Y (0 = Non-Toxic, 1 = Toxic):
is_toxic
0    143346
1     16225
Name: count, dtype: int64


TF-IDF

In [10]:
from features.tfidf_extractor import TextVectorizer
from sklearn.model_selection import train_test_split

#pake stop word
X1_train, X1_test, Y_train, Y_test = train_test_split(X1, Y, test_size=0.2, random_state=42)

vectorizer = TextVectorizer(max_features=5000)
X1_train_tfidf = vectorizer.fit_transform(X1_train)
X1_test_tfidf = vectorizer.transform(X1_test)

print("Shape X1_train TF-IDF:", X1_train_tfidf.shape)

#non stopword
X2_train, X2_test, Y_train, Y_test = train_test_split(X2, Y, test_size=0.2, random_state=42)

vectorizer = TextVectorizer(max_features=5000)
X2_train_tfidf = vectorizer.fit_transform(X2_train)
X2_test_tfidf = vectorizer.transform(X2_test)

print("Shape X2_train TF-IDF:", X2_train_tfidf.shape)

Shape X1_train TF-IDF: (127656, 5000)
Shape X2_train TF-IDF: (127656, 5000)


LOGISTIC REGRRESSION (MODEL)

In [11]:
from models.logistic_model import ToxicClassifier

classifier = ToxicClassifier()

#stopword
classifier.train(X1_train_tfidf, Y_train)

#nonstopword
classifier.train(X2_train_tfidf, Y_train)

#stopword
Y_pred = classifier.predict(X1_test_tfidf)
#nonstopword
Y2_pred = classifier.predict(X2_test_tfidf)

#hasil
report1 = classifier.evaluate(Y_test, Y_pred)
report2 = classifier.evaluate(Y_test, Y2_pred)

print("STOP WORD")
print(report1)
print("NON STOP WORD")
print(report2)

STOP WORD
              precision    recall  f1-score   support

           0       0.90      0.99      0.94     28671
           1       0.28      0.04      0.08      3244

    accuracy                           0.89     31915
   macro avg       0.59      0.52      0.51     31915
weighted avg       0.84      0.89      0.85     31915

NON STOP WORD
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28671
           1       0.92      0.63      0.75      3244

    accuracy                           0.96     31915
   macro avg       0.94      0.81      0.86     31915
weighted avg       0.96      0.96      0.95     31915

